In [57]:
!docker exec -it module-7-solution-redpanda-1 rpk --version

rpk version (Redpanda CLI): v25.3.9 (rev 836b4a36ef6d5121edbb1e68f0f673c2a8a244e2)


In [1]:
!docker exec -it module-7-solution-redpanda-1 rpk topic create green-trips

TOPIC        STATUS
green-trips  OK


In [2]:
import pandas as pd

file_path = "green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime','lpep_dropoff_datetime','PULocationID', 'DOLocationID','passenger_count','trip_distance', 'tip_amount','total_amount']
df = pd.read_parquet(file_path, columns=columns,engine='fastparquet')
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [3]:
df

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12
...,...,...,...,...,...,...,...,...
49411,2025-10-31 23:02:00,2025-11-01 00:28:33,241,61,NaN,20.09,0.00,63.84
49412,2025-10-31 23:51:34,2025-11-01 00:20:58,53,225,NaN,10.11,0.00,34.76
49413,2025-10-31 23:08:00,2025-10-31 23:42:00,7,170,NaN,4.20,10.03,60.17
49414,2025-10-31 23:45:00,2025-11-01 00:08:00,255,25,NaN,4.20,4.86,37.29


In [4]:
from dataclasses import dataclass

In [5]:
@dataclass
class GreenTrip:
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    PULocationID: int
    DOLocationID: int
    passenger_count: int
    trip_distance: float
    tip_amount: float
    total_amount: float

In [6]:
def green_trip_from_row(row):
    return GreenTrip(
        lpep_pickup_datetime=str(row['lpep_pickup_datetime']) ,
        lpep_dropoff_datetime=str(row['lpep_dropoff_datetime']) ,
        PULocationID=int(row['PULocationID']) ,
        DOLocationID=int(row['DOLocationID']) ,
        passenger_count=int(row['passenger_count']) if not pd.isna(row['passenger_count']) else 1,
        trip_distance=float(row['trip_distance']) ,
        tip_amount=float(row['tip_amount']) ,
        total_amount=float(row['total_amount']) ,
    )

In [7]:
green_trip = green_trip_from_row(df.iloc[-1])
green_trip

GreenTrip(lpep_pickup_datetime='2025-10-31 23:23:00', lpep_dropoff_datetime='2025-10-31 23:37:00', PULocationID=195, DOLocationID=33, passenger_count=1, trip_distance=3.0, tip_amount=0.0, total_amount=19.6)

In [8]:
import json
from kafka import KafkaProducer


server = 'localhost:9092'


In [9]:
import dataclasses

topic_name = 'green-trips'

def green_trip_serializer(green_trip):
    green_trip_dict = dataclasses.asdict(green_trip)
    json_str = json.dumps(green_trip_dict)
    return json_str.encode('utf-8')

In [10]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=green_trip_serializer
)

In [11]:
import time

t0 = time.time()

for _, row in df.iterrows():
    green_trip = green_trip_from_row(row)
    producer.send(topic_name, value=green_trip)
    #print(f"Sent: {green_trip}")
    #time.sleep(0.001)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 9.76 seconds


In [ ]:
def green_trip_deserializer(data):
    json_str = data.decode('utf-8')
    green_trip_dict = json.loads(json_str)
    return GreenTrip(**green_trip_dict)

In [ ]:
from kafka import  KafkaConsumer
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='green-trip-counter-group',
    value_deserializer=green_trip_deserializer
)

In [ ]:
print(f"Listening to  {topic_name}...")

trip_count = 0

try:
    for message in consumer:
        green_trip_data = message.value
        
        if green_trip_data.trip_distance > 5.0:
            trip_count += 1
            print(f"Current count of trips > 5.0km: {trip_count}")
    
except KeyboardInterrupt:
    print("Stopping consumer...")
    
    consumer.close()

In [13]:
!docker compose exec jobmanager ./bin/flink run \
    -py /opt/src/gt_pickup_location_job.py \
    --pyFiles /opt/src -d

Starting batch job processing...
Job has been submitted with JobID 69cccbe1a81982f93c46ad5de30dcfb4
Job completed successfully. You can now query the PostgreSQL table.


Q4 - 74

In [65]:
!docker exec -it module-7-solution-redpanda-1 rpk topic delete green-trips && docker exec -it module-7-solution-redpanda-1 rpk topic create green-trips


TOPIC        STATUS
green-trips  OK
TOPIC        STATUS
green-trips  OK


In [66]:
import time
t0 = time.time()

for _, row in df.iterrows():
    green_trip = green_trip_from_row(row)
    producer.send(topic_name, value=green_trip)
    #print(f"Sent: {green_trip}")
    #time.sleep(0.001)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 9.78 seconds


In [67]:
!docker compose exec jobmanager ./bin/flink run \
    -py /opt/src/gt_longest_streak_job.py \
    --pyFiles /opt/src -d

Submitting Flink job → http://localhost:8081
Job has been submitted with JobID 702e4e1aa69cb1b9813f278b133ab9ff


Q5 -81

In [69]:
!docker exec -it module-7-solution-redpanda-1 rpk topic delete green-trips && docker exec -it module-7-solution-redpanda-1 rpk topic create green-trips


TOPIC        STATUS
green-trips  OK
TOPIC        STATUS
green-trips  OK


In [70]:
t0 = time.time()

for _, row in df.iterrows():
    green_trip = green_trip_from_row(row)
    producer.send(topic_name, value=green_trip)
    #print(f"Sent: {green_trip}")
    #time.sleep(0.001)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 9.96 seconds


In [72]:
!docker compose exec jobmanager ./bin/flink run \
    -py /opt/src/gt_largest_tip_job.py \
    --pyFiles /opt/src -d

Job has been submitted with JobID 50e5d1ba4a457e8276609b786a25c436
^C

------------------------------------------------------------
 The program finished with the following exception:

org.apache.flink.client.program.ProgramInvocationException: The main method caused an error: Shutdown in progress
	at org.apache.flink.client.program.PackagedProgram.callMainMethod(PackagedProgram.java:360)
	at org.apache.flink.client.program.PackagedProgram.invokeInteractiveModeForExecution(PackagedProgram.java:223)
	at org.apache.flink.client.ClientUtils.executeProgram(ClientUtils.java:105)
	at org.apache.flink.client.cli.CliFrontend.executeProgram(CliFrontend.java:1017)
	at org.apache.flink.client.cli.CliFrontend.run(CliFrontend.java:230)
	at org.apache.flink.client.cli.CliFrontend.parseAndRun(CliFrontend.java:1261)
	at org.apache.flink.client.cli.CliFrontend.lambda$mainInternal$11(CliFrontend.java:1355)
	at org.apache.flink.runtime.security.contexts.NoOpSecurityContext.runSecured(NoOpSecurityContext.

Q6 - 2025-10-16 18:00:00.000